In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from src.calibration import IsotonicCalibrator, PlattScaling
from src.data import DATASETS, RANDOM_STATE, load_splits
from src.metrics import evaluate

RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

MODELS = {
    "LR": lambda C: LogisticRegression(C=C, max_iter=1000),
    "RF": lambda min_samples_leaf: RandomForestClassifier(
        n_estimators=200, max_features="sqrt", min_samples_leaf=min_samples_leaf,
        n_jobs=-1, random_state=RANDOM_STATE),
}

In [2]:
data = {}
for name in DATASETS:
    X_train, X_cal, X_test, y_train, y_cal, y_test = load_splits(name)
    data[name] = {"train": (X_train, y_train), "cal": (X_cal, y_cal), "test": (X_test, y_test)}

In [3]:
cv_results = pd.read_csv(RESULTS_DIR / "cv_results.csv")
best = cv_results.loc[cv_results.groupby(["dataset", "model"], sort=False)["val_log_loss"].idxmin()]

hyperparams = {}
for name, model_name, param, value in best[["dataset", "model", "param", "value"]].itertuples(index=False):
    hyperparams[name, model_name] = {param: int(value) if param == "min_samples_leaf" else value}

In [4]:
models = {}
for name in DATASETS:
    X, y = data[name]["train"]
    for model_name, make in MODELS.items():
        models[name, model_name] = make(**hyperparams[name, model_name]).fit(X,y)
        print(f"{name:10s} {model_name}: {models[name, model_name]}")

higgs      LR: LogisticRegression(C=0.1, max_iter=1000)
higgs      RF: RandomForestClassifier(min_samples_leaf=5, n_estimators=200, n_jobs=-1,
                       random_state=42)
covertype  LR: LogisticRegression(C=100.0, max_iter=1000)
covertype  RF: RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
adult      LR: LogisticRegression(C=0.1, max_iter=1000)
adult      RF: RandomForestClassifier(min_samples_leaf=2, n_estimators=200, n_jobs=-1,
                       random_state=42)


In [5]:
outputs = {}
for (name, model_name), model in models.items():
    for split in ("cal", "test"):
        X = data[name][split][0]
        proba = model.predict_proba(X)[:,1]
        score = model.decision_function(X) if model_name =="LR" else proba
        outputs[name, model_name, split] = (score, proba)

np.savez_compressed(
    RESULTS_DIR / "uncalibrated_outputs.npz",
    **{f"{n}_{m}_{s}_{kind}": arr
       for (n, m, s), pair in outputs.items() for kind, arr in zip(("score", "proba"), pair)})